In [0]:
# =========================================
# IMPORTS
# =========================================

import re

# =========================================
# CONFIGURATION
# =========================================

base_path = "s3://retail-sales-datawarehouse"

zones = [
    "bronze",
    "processed"
]

entities = [
    "customers",
    "products",
    "stores",
    "sales"
]

# =========================================
# FILE NAMING CONVENTION
# Example:
# customers_06052026_101500.csv
# =========================================

pattern = r'.*_\d{14}\.csv$'

# =========================================
# START ARCHIVAL PROCESS
# =========================================

for zone in zones:

    print(f"\n========== PROCESSING ZONE: {zone.upper()} ==========")

    for entity in entities:

        source_path = f"{base_path}/{zone}/{entity}/"

        archive_path = f"{base_path}/archive/{zone}/{entity}/"

        print(f"\nChecking path: {source_path}")

        try:

            # =========================================
            # READ FILES
            # =========================================

            files = dbutils.fs.ls(source_path)

            # =========================================
            # FILTER VALID FILES
            # =========================================

            valid_files = []

            for file in files:

                if re.match(pattern, file.name):

                    valid_files.append(file)

            # =========================================
            # VALIDATION: FILES EXIST
            # =========================================

            if len(valid_files) == 0:

                print("No valid files found")

                continue

            # =========================================
            # SORT FILES
            # Latest timestamp file last
            # =========================================

            valid_files = sorted(
                valid_files,
                key=lambda x: x.name
            )

            latest_file = valid_files[-1]

            old_files = valid_files[:-1]

            print(f"Latest active file retained: {latest_file.name}")

            # =========================================
            # MOVE OLD FILES TO ARCHIVE
            # =========================================

            for old_file in old_files:

                destination = archive_path + old_file.name

                dbutils.fs.mv(
                    old_file.path,
                    destination
                )

                print(f"Archived file: {old_file.name}")

            # =========================================
            # FINAL STATUS
            # =========================================

            print(f"Archival completed for {entity}")

        except Exception as e:

            print(f"Error processing {entity}")

            print(str(e))

# =========================================
# ARCHIVAL PROCESS COMPLETE
# =========================================

print("\n===================================")
print("ARCHIVAL PROCESS COMPLETED")
print("===================================")